# Brasil 2040 — PDF Embedding Pipeline
**Model:** `intfloat/multilingual-e5-large`  
**Storage:** Supabase (pgvector)  
**Dataset:** `brasil2040` (42 PDFs in Portuguese)

### Steps
1. Install dependencies
2. Extract text from all 42 PDFs
3. Clean and chunk the text
4. Generate embeddings with `multilingual-e5-large`
5. Save locally (backup)
6. Upload to Supabase via pgvector

## Step 1 — Install Dependencies

In [ ]:
!pip install sentence-transformers pymupdf supabase tqdm langdetect

## Step 2 — Configuration

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

# These names must match exactly what you typed in Add-ons → Secrets
SUPABASE_URL = secrets.get_secret("SUPABASE_URL")
SUPABASE_KEY = secrets.get_secret("SUPABASE_KEY")

PDF_DIR    = "/kaggle/input/datasets/punkada/brasil2040"
MODEL_NAME = "/kaggle/input/models/gsasikiran/multilingual-e5-large/transformers/version-1/1"
CHUNK_SIZE    = 600
CHUNK_OVERLAP = 100
EMBED_DIM     = 1024
BATCH_SIZE    = 32
TABLE_NAME    = "documents"

print("✅ Config loaded")
print(f"   PDF dir : {PDF_DIR}")
print(f"   Model   : {MODEL_NAME}")
print(f"   Table   : {TABLE_NAME}")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input/datasets/punkada/brasil2040"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ── DIAGNOSTIC: run this BEFORE cell 4 ───────────────────────────────────────
import fitz
import re
import pathlib

TERMS_TO_CHECK = ["16,7", "74%", "99%", "8,5", "2,5", "5,5", "67,5", "112,3", "34%", "20,6"]

OLD_CLEAN_PATTERN = re.compile(r'\b\w{1,2}\b')

problem_files = {}

for pdf_path in sorted(pathlib.Path(PDF_DIR).glob("*.pdf")):
    doc = fitz.open(str(pdf_path))
    fname = pdf_path.name
    file_issues = []
    
    for page_num, page in enumerate(doc, 1):
        raw = page.get_text("text")
        after_clean = OLD_CLEAN_PATTERN.sub('', raw)
        
        for term in TERMS_TO_CHECK:
            in_raw     = term in raw
            in_cleaned = term in after_clean
            
            if in_raw and not in_cleaned:
                file_issues.append({
                    "page": page_num,
                    "term": term,
                    "cause": "destroyed_by_regex",
                    "context": raw[max(0, raw.index(term)-40) : raw.index(term)+40].replace('\n',' ')
                })
            elif not in_raw:
                # Check if it's in a table that get_text("text") misses
                try:
                    tabs = page.find_tables()
                    for tab in tabs:
                        tab_text = str(tab.extract())
                        if term in tab_text:
                            file_issues.append({
                                "page": page_num,
                                "term": term,
                                "cause": "only_in_table",
                                "context": tab_text[max(0, tab_text.index(term)-40) : tab_text.index(term)+40]
                            })
                except Exception:
                    pass
    
    doc.close()
    if file_issues:
        problem_files[fname] = file_issues

# ── Report ────────────────────────────────────────────────────────────────────
if not problem_files:
    print("✅ No issues found — all terms survive cleaning in all PDFs")
else:
    print(f"⚠️  Problems found in {len(problem_files)} file(s):\n")
    for fname, issues in problem_files.items():
        print(f"  📄 {fname}")
        for issue in issues:
            icon = "❌" if issue["cause"] == "destroyed_by_regex" else "📊"
            print(f"    {icon} Page {issue['page']} | '{issue['term']}' | cause: {issue['cause']}")
            print(f"       Context: ...{issue['context']}...")
        print()

## Step 3 — Extract Text from PDFs

In [ ]:
import fitz
import pathlib
import re
from tqdm import tqdm

def clean_text(text: str) -> str:
    text = re.sub(r'[^\w\s\.,;:!?\-()\[\]%°/€$àáâãéêíóôõúüçñÀÁÂÃÉÊÍÓÔÕÚÜÇÑ]', ' ', text)
    text = re.sub(r'([a-z])-\n([a-z])', r'\1\2', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def extract_pdf(pdf_path: str) -> list[dict]:
    """
    Extract text page-by-page, also pulling table content explicitly.
    """
    pages = []
    try:
        doc = fitz.open(pdf_path)
        filename = pathlib.Path(pdf_path).name
        
        for page_num, page in enumerate(doc, start=1):
            # Get regular text
            raw = page.get_text("text")
            
            # Also extract tables as structured text
            # This catches numbers that live in table cells
            table_text = ""
            try:
                tabs = page.find_tables()
                for tab in tabs:
                    df = tab.to_pandas()
                    # Convert table to readable text: "Header: value, header2: value2"
                    for _, row in df.iterrows():
                        row_str = " | ".join(
                            str(v) for v in row.values 
                            if v is not None and str(v).strip() not in ("", "None")
                        )
                        if row_str.strip():
                            table_text += row_str + "\n"
            except Exception:
                pass  # find_tables() not available in all PyMuPDF versions
            
            combined = raw + ("\n\n[TABELA]\n" + table_text if table_text else "")
            cleaned = clean_text(combined)
            
            if len(cleaned) > 50:
                pages.append({
                    "file": filename,
                    "page": page_num,
                    "text": cleaned
                })
        doc.close()
        
    except Exception as e:
        print(f"  ⚠️  Error reading {pdf_path}: {e}")
    return pages

# ── Run extraction ─────────────────────────────────────────────────────────────
pdf_files = sorted(pathlib.Path(PDF_DIR).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files\n")

all_pages = []
for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    all_pages.extend(extract_pdf(str(pdf_path)))

total_chars = sum(len(p["text"]) for p in all_pages)
print(f"\n✅ Extracted {len(all_pages)} pages from {len(pdf_files)} PDFs")
print(f"   Total characters: {total_chars:,}")
print(f"   Approx. text size: {total_chars / 1_000_000:.1f} MB")

## Step 4 — Chunk Text

In [ ]:
def chunk_page(page: dict, chunk_size: int, overlap: int) -> list[dict]:
    """
    Split on sentence boundaries when possible, fall back to word windows.
    This prevents cutting "16," from "7 bilhões" across chunk boundaries.
    """
    text = page["text"]
    
    # Try sentence-aware splitting first
    # Split on ". " or ".\n" but not on decimal points like "16,7"
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-ZÁÀÂÃÉÊÍÓÔÕÚ])', text)
    
    chunks = []
    current = []
    current_len = 0
    chunk_idx = 0
    
    for sent in sentences:
        words = sent.split()
        if current_len + len(words) > chunk_size and current:
            chunk_text = " ".join(current)
            if len(current) >= 30:
                chunks.append({
                    "file":       page["file"],
                    "page":       page["page"],
                    "chunk_idx":  chunk_idx,
                    "text":       chunk_text,
                    "text_embed": "passage: " + chunk_text,
                })
                chunk_idx += 1
            # Keep overlap: last N words carry over
            overlap_words = current[-overlap:] if len(current) > overlap else current
            current = overlap_words + words
            current_len = len(current)
        else:
            current.extend(words)
            current_len += len(words)
    
    # Don't forget the last chunk
    if len(current) >= 30:
        chunk_text = " ".join(current)
        chunks.append({
            "file":       page["file"],
            "page":       page["page"],
            "chunk_idx":  chunk_idx,
            "text":       chunk_text,
            "text_embed": "passage: " + chunk_text,
        })
    
    return chunks

# ── Run chunking ───────────────────────────────────────────────────────────────
all_chunks = []
for page in all_pages:
    all_chunks.extend(chunk_page(page, CHUNK_SIZE, CHUNK_OVERLAP))

avg_words = sum(len(c["text"].split()) for c in all_chunks) / len(all_chunks)
print(f"✅ Generated {len(all_chunks):,} chunks")
print(f"   Average chunk length: {avg_words:.0f} words")
print(f"   Estimated embeddings size: {len(all_chunks) * EMBED_DIM * 4 / 1_000_000:.1f} MB (float32)")

# Preview
print("\n── Sample chunk ────────────────────────────────────")
sample = all_chunks[10]
print(f"File : {sample['file']}")
print(f"Page : {sample['page']}  |  Chunk idx: {sample['chunk_idx']}")
print(f"Text : {sample['text'][:200]}...")

## Step 5 — Load Model & Generate Embeddings

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\nLoading {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME, device=device)
print("✅ Model loaded")

In [ ]:
# Encode all chunks — uses the prefixed text ("passage: ...")
texts_to_embed = [c["text_embed"] for c in all_chunks]

print(f"Encoding {len(texts_to_embed):,} chunks in batches of {BATCH_SIZE} ...")
embeddings = model.encode(
    texts_to_embed,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,   # cosine similarity works with dot product after this
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f"\n✅ Embeddings shape : {embeddings.shape}")
print(f"   dtype            : {embeddings.dtype}")
print(f"   Total size       : {embeddings.nbytes / 1_000_000:.1f} MB")

## Step 6 — Save Locally (Backup)

In [ ]:
import json

# Save embeddings as .npy
np.save("/kaggle/working/embeddings.npy", embeddings)

# Save metadata as JSON (without the text_embed prefix field)
metadata = [{
    "file":      c["file"],
    "page":      c["page"],
    "chunk_idx": c["chunk_idx"],
    "text":      c["text"],
} for c in all_chunks]

with open("/kaggle/working/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ Saved:")
print("   /kaggle/working/embeddings.npy")
print("   /kaggle/working/metadata.json")

## Step 7 — Set Up Supabase Table

Before running the upload cell, run this SQL once in your **Supabase SQL Editor**:

```sql
-- Enable the pgvector extension
create extension if not exists vector;

-- Create the documents table
create table if not exists documents (
  id         bigserial primary key,
  file       text not null,
  page       integer not null,
  chunk_idx  integer not null,
  text       text not null,
  embedding  vector(1024)
);

-- Create an HNSW index for fast similarity search
create index if not exists documents_embedding_idx
  on documents
  using hnsw (embedding vector_cosine_ops);

-- Create a search function to call from your backend
create or replace function match_documents (
  query_embedding vector(1024),
  match_count     int default 5
)
returns table (
  id        bigint,
  file      text,
  page      integer,
  text      text,
  similarity float
)
language sql stable
as $$
  select
    id, file, page, text,
    1 - (embedding <=> query_embedding) as similarity
  from documents
  order by embedding <=> query_embedding
  limit match_count;
$$;
```

## Step 8 — Upload to Supabase

In [ ]:
from supabase import create_client, Client
import time

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Connected to Supabase")

UPLOAD_BATCH = 100   # rows per insert — Supabase handles this well

def upload_batch(rows: list[dict]):
    """Insert a batch of rows into Supabase."""
    response = supabase.table(TABLE_NAME).insert(rows).execute()
    return response

# Build rows
rows_to_insert = []
for chunk, emb in zip(metadata, embeddings):
    rows_to_insert.append({
        "file":      chunk["file"],
        "page":      chunk["page"],
        "chunk_idx": chunk["chunk_idx"],
        "text":      chunk["text"],
        "embedding": emb.tolist(),   # Supabase expects a plain list
    })

print(f"Uploading {len(rows_to_insert):,} rows in batches of {UPLOAD_BATCH} ...")

errors = []
for i in tqdm(range(0, len(rows_to_insert), UPLOAD_BATCH), desc="Uploading"):
    batch = rows_to_insert[i : i + UPLOAD_BATCH]
    try:
        upload_batch(batch)
    except Exception as e:
        errors.append((i, str(e)))
        print(f"  ⚠️  Batch {i}–{i+UPLOAD_BATCH} failed: {e}")
        time.sleep(2)  # back off briefly on error

if errors:
    print(f"\n⚠️  {len(errors)} batches failed. Check errors list.")
else:
    print(f"\n✅ All {len(rows_to_insert):,} rows uploaded successfully!")

## Step 9 — Verify Upload & Test Search

In [ ]:
# Count rows in table
count_result = supabase.table(TABLE_NAME).select("id", count="exact").execute()
print(f"✅ Rows in Supabase table '{TABLE_NAME}': {count_result.count:,}")

# Test a semantic search query
# Note: queries MUST use the "query: " prefix (different from "passage: ")
test_query = "query: Quais são os objetivos do Brasil para 2040?"
query_embedding = model.encode(
    [test_query],
    normalize_embeddings=True,
    convert_to_numpy=True
)[0].tolist()

result = supabase.rpc(
    "match_documents",
    {"query_embedding": query_embedding, "match_count": 3}
).execute()

print(f"\n── Top 3 results for: '{test_query[7:]}' ──")
for i, row in enumerate(result.data, 1):
    print(f"\n[{i}] {row['file']} — page {row['page']}")
    print(f"    Similarity : {row['similarity']:.4f}")
    print(f"    Preview    : {row['text'][:200]}...")

## ✅ Done!

Your pipeline is complete. Here's what was built:

| Step | Result |
|------|--------|
| PDF extraction | Text from all 42 PDFs, page by page |
| Chunking | ~400-word overlapping chunks with metadata |
| Embeddings | 1024-dim vectors via `multilingual-e5-large` |
| Storage | Uploaded to Supabase with HNSW index |
| Search function | `match_documents()` ready for your backend |

### How to query from your backend (Python example)
```python
from sentence_transformers import SentenceTransformer
from supabase import create_client

model = SentenceTransformer("intfloat/multilingual-e5-large")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def search(user_question: str, top_k: int = 5):
    # IMPORTANT: use "query: " prefix for questions
    embedding = model.encode(
        ["query: " + user_question],
        normalize_embeddings=True
    )[0].tolist()

    results = supabase.rpc(
        "match_documents",
        {"query_embedding": embedding, "match_count": top_k}
    ).execute()

    return results.data  # list of {file, page, text, similarity}
```